# Study 865 — Credit → Equity Lead-Lag 🔗

**Does high-yield credit really *turn before* stocks?**

Desk lore says "credit leads equity": high-yield credit, measured *duration-hedged* —
**HYG in excess of IEF** — is supposed to inflect a beat ahead of the stock market. The
self-contained, testable version is a **Granger-style lead-lag**: does the trailing 1-4-week
HY-excess return **predict the NEXT week's SPY return**? We take the weekly form on the four
ETFs (2007-05-01 → 2026-06-30, 4,822 daily rows → 1,001 weekly
closes) and ask the two honest questions: does the credit trend **lead** the equity leg, and
can a **costed** SPY↔IEF overlay beat buy-and-hold?

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Distinct from study 115 (credit-spread **level** warning), 832 (HY
momentum as a daily sign-timer), 131 (utilities canary), 379 (generic ETF lead-lag).*


## 1. The idea in one picture

When high-yield bonds out-earn duration-matched Treasuries, the market is paying *up* for credit risk — risk appetite is rising. The folk rule: because credit turns *first*, a strong trailing HY-excess week should foreshadow a *strong SPY week to come*. So we regress **next** week's SPY return on the trailing 4-week credit trend. If credit leads, the slope is positive. Is it?

In [1]:
R = dict(beta=-0.0546, beta_t=-1.7, per_sd_bps=-19.6, r2=0.601, on_bps=19.5, off_bps=27.0, diff_bps=-7.5, disc_t=-0.48, on_frac=0.582)
print('predictive regression  r_SPY[t+1] ~ trailing 4-week HY-excess trend[t]:')
print('  slope           : %+.4f  (SPY move %+.1f bps per 1-sigma credit trend)' % (R['beta'], R['per_sd_bps']))
print('  Newey-West t     : %+.2f   R2 = %.2f%%' % (R['beta_t'], R['r2']))
print('  -> WRONG SIGN and insignificant: credit up does NOT foreshadow stocks up')
print()
print('risk-on weeks (trend>0, %.0f%% of weeks): next-week SPY %+.1f bps' % (R['on_frac']*100, R['on_bps']))
print('risk-off weeks                     : next-week SPY %+.1f bps' % R['off_bps'])
print('difference (on - off)              : %+.1f bps/week  (NW t = %+.2f)' % (R['diff_bps'], R['disc_t']))

predictive regression  r_SPY[t+1] ~ trailing 4-week HY-excess trend[t]:
  slope           : -0.0546  (SPY move -19.6 bps per 1-sigma credit trend)
  Newey-West t     : -1.70   R2 = 0.60%
  -> WRONG SIGN and insignificant: credit up does NOT foreshadow stocks up

risk-on weeks (trend>0, 58% of weeks): next-week SPY +19.5 bps
risk-off weeks                     : next-week SPY +27.0 bps
difference (on - off)              : -7.5 bps/week  (NW t = -0.48)


## 2. Is the machinery even able to see a lead? A live synthetic control

We plant a real one-week lead of credit over equity in a seeded toy world (`edge>0`: the risk-on factor drives SPY *five trading days later*) and check the detector recovers it — and stays *silent* on the null (`edge=0`, credit trend present but leads nothing). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from credit_lead import data, strategy as st
null_t = np.array([st.leadlag_regression(data.synthetic_panel(edge=0.0, seed=865+s, n_days=3000), 4)['beta_t_nw'] for s in range(6)])
planted = st.leadlag_regression(data.synthetic_panel(edge=0.02, seed=865, n_days=3000), 4)
print('null worlds  : regression NW t mean %+.2f over 6 seeds  (should be ~0)' % null_t.mean())
print('planted world: regression NW t = %+.2f  (should light up, positive)' % planted['beta_t_nw'])

null worlds  : regression NW t mean -0.06 over 6 seeds  (should be ~0)
planted world: regression NW t = +26.69  (should light up, positive)


## 3. The honest verdict — credit does *not* lead equity here

On the real tape the trailing credit trend predicts next-week SPY with the **wrong sign**: a stronger credit week foreshadows a slightly *weaker* SPY week (slope -0.0546, i.e. -19.6 bps per 1σ trend, NW *t* = **-1.70**), and the risk-on−risk-off next-week difference (-7.5 bps) sits just -0.4σ inside a label-shift placebo (p = 0.66). The sign is **consistently wrong** across both eras. The costed SPY↔IEF overlay never beats buy-and-hold (net Sharpe 0.644 vs 0.645 at 1 bp, giving up 3.3%/yr of return) — it only trims the drawdown (-33% vs -55%). **Signal: None. Tradability: Mirage.**